In [ ]:
# import scanpy as sc
# zhu_adata = sc.read_h5ad('/home/ec2-user/scitoseq/data/GWCD4i.DE_stats.h5ad')
# zhu_adata

In [3]:
import h5py
import numpy as np

filepath = '/home/ec2-user/scitoseq/data/GWCD4i.DE_stats.h5ad'

with h5py.File(filepath, 'r') as f:
    
    print("=== obs group contents ===")
    obs = f['obs']
    for key in list(obs.keys())[:15]:
        item = obs[key]
        if isinstance(item, h5py.Dataset):
            print(f"  {key}: shape={item.shape}, dtype={item.dtype}")
            # Show first few values
            try:
                vals = item[:3]
                if item.dtype.kind == 'S' or item.dtype.kind == 'O':
                    vals = [v.decode() if isinstance(v, bytes) else v for v in vals]
                print(f"    first 3: {vals}")
            except:
                pass
        else:
            print(f"  {key}: Group with keys {list(item.keys())[:5]}")
    
    print("\n=== var group contents ===")
    var = f['var']
    for key in list(var.keys())[:15]:
        item = var[key]
        if isinstance(item, h5py.Dataset):
            print(f"  {key}: shape={item.shape}, dtype={item.dtype}")
            try:
                vals = item[:3]
                if item.dtype.kind == 'S' or item.dtype.kind == 'O':
                    vals = [v.decode() if isinstance(v, bytes) else v for v in vals]
                print(f"    first 3: {vals}")
            except:
                pass
        else:
            print(f"  {key}: Group with keys {list(item.keys())[:5]}")
    
    print("\n=== layers ===")
    if 'layers' in f:
        layers = f['layers']
        for key in layers.keys():
            item = layers[key]
            if isinstance(item, h5py.Dataset):
                print(f"  {key}: shape={item.shape}, dtype={item.dtype}")
            else:
                print(f"  {key}: Group with keys {list(item.keys())[:5]}")
    
    print("\n=== X matrix ===")
    X = f['X']
    print(f"  Type: {type(X)}")
    print(f"  Shape: {X.shape if hasattr(X, 'shape') else 'checking...'}")
    print(f"  Dtype: {X.dtype if hasattr(X, 'dtype') else 'N/A'}")

=== obs group contents ===
  chunk: Group with keys ['categories', 'codes']
  culture_condition: Group with keys ['categories', 'codes']
  index: shape=(33983,), dtype=object
    first 3: ['ENSG00000121410_Rest', 'ENSG00000121410_Stim48hr', 'ENSG00000121410_Stim8hr']
  n_cells_target: shape=(33983,), dtype=float64
    first 3: [734. 825. 753.]
  n_down_genes: shape=(33983,), dtype=int64
    first 3: [0 0 0]
  n_downstream: shape=(33983,), dtype=int64
    first 3: [1 0 1]
  n_total_de_genes: shape=(33983,), dtype=int64
    first 3: [1 0 1]
  n_total_genes_category: Group with keys ['categories', 'codes']
  n_up_genes: shape=(33983,), dtype=int64
    first 3: [1 0 1]
  offtarget_flag: shape=(33983,), dtype=bool
    first 3: [False False False]
  ontarget_effect_category: Group with keys ['categories', 'codes']
  ontarget_effect_size: shape=(33983,), dtype=float64
    first 3: [0. 0. 0.]
  ontarget_significant: shape=(33983,), dtype=bool
    first 3: [False False False]
  target_baseMean:

In [4]:
import h5py
import pandas as pd
import numpy as np

filepath = '/home/ec2-user/scitoseq/data/GWCD4i.DE_stats.h5ad'

with h5py.File(filepath, 'r') as f:
    
    # Get gene names from var
    gene_names = [x.decode() if isinstance(x, bytes) else x for x in f['var']['gene_name'][:]]
    gene_ids = [x.decode() if isinstance(x, bytes) else x for x in f['var']['_index'][:]]
    
    print(f"Total measured genes: {len(gene_names)}")
    
    # Find IL2RA
    if 'IL2RA' in gene_names:
        il2ra_idx = gene_names.index('IL2RA')
        print(f"IL2RA found at index {il2ra_idx}")
        print(f"IL2RA ENSEMBL ID: {gene_ids[il2ra_idx]}")
    else:
        print("IL2RA not found by name, checking ENSEMBL ID...")
        if 'ENSG00000134460' in gene_ids:
            il2ra_idx = gene_ids.index('ENSG00000134460')
            print(f"IL2RA (ENSG00000134460) found at index {il2ra_idx}")
        else:
            print("IL2RA not found!")
            il2ra_idx = None
    
    if il2ra_idx is not None:
        # Get perturbation info from obs
        obs_index = [x.decode() if isinstance(x, bytes) else x for x in f['obs']['index'][:]]
        
        # Extract IL2RA column from log_fc layer
        print("\nExtracting IL2RA trans-effects...")
        il2ra_lfc = f['layers']['log_fc'][:, il2ra_idx]
        il2ra_zscore = f['layers']['zscore'][:, il2ra_idx]
        il2ra_pval = f['layers']['adj_p_value'][:, il2ra_idx]
        
        # Create dataframe
        zhu_il2ra_df = pd.DataFrame({
            'perturbation_id': obs_index,
            'il2ra_lfc': il2ra_lfc,
            'il2ra_zscore': il2ra_zscore,
            'il2ra_adj_pval': il2ra_pval
        })
        
        # Parse perturbation ID to get gene and condition
        zhu_il2ra_df['perturbed_gene'] = zhu_il2ra_df['perturbation_id'].str.replace('_Rest|_Stim8hr|_Stim48hr', '', regex=True)
        zhu_il2ra_df['condition'] = zhu_il2ra_df['perturbation_id'].str.extract('_(Rest|Stim8hr|Stim48hr)$')[0]
        
        print(f"\nExtracted {len(zhu_il2ra_df)} trans-effects on IL2RA")
        print(f"\nConditions: {zhu_il2ra_df['condition'].value_counts().to_dict()}")
        
        # Preview
        print("\nTop 10 positive regulators (Rest condition):")
        print(zhu_il2ra_df[zhu_il2ra_df['condition'] == 'Rest'].nsmallest(10, 'il2ra_lfc')[
            ['perturbed_gene', 'il2ra_lfc', 'il2ra_zscore', 'il2ra_adj_pval']
        ])
        
        # Save
        zhu_il2ra_df.to_csv('/home/ec2-user/scitoseq/data/zhu_il2ra_trans_effects.csv', index=False)
        print("\nSaved to zhu_il2ra_trans_effects.csv")

Total measured genes: 10282
IL2RA found at index 4345
IL2RA ENSEMBL ID: ENSG00000134460

Extracting IL2RA trans-effects...

Extracted 33983 trans-effects on IL2RA

Conditions: {'Stim8hr': 11415, 'Rest': 11287, 'Stim48hr': 11281}

Top 10 positive regulators (Rest condition):
        perturbed_gene  il2ra_lfc  il2ra_zscore  il2ra_adj_pval
13367  ENSG00000134460  -1.979786     -6.938083    4.923279e-08
12027  ENSG00000112339  -1.785498     -3.056353    1.258354e-01
32414  ENSG00000172262  -1.671853     -3.338749    2.774246e-01
12953  ENSG00000149428  -1.648765     -2.427282    4.694422e-01
8411   ENSG00000114867  -1.612224     -3.594076    9.225215e-03
24762  ENSG00000106803  -1.607716     -3.221143    3.427687e-02
30459  ENSG00000126261  -1.568479     -3.368251    3.706687e-02
11515  ENSG00000030582  -1.555138     -3.343942    3.785385e-01
30542  ENSG00000103275  -1.478337     -4.137052    4.931465e-03
2655   ENSG00000163930  -1.474774     -3.798904    4.387619e-02

Saved to zhu_il2ra_t

In [1]:
#!/usr/bin/env python3
"""
Extract IL2 and IFNG trans-effects from Zhu et al. perturb-seq h5ad file
for comparison with Schmidt et al. CRISPRa FACS screen

This script extracts the trans-effect measurements (LFC, z-score, p-value)
for IL2 and IFNG across all gene perturbations and conditions.

Usage:
    python extract_zhu_il2_ifng_transeffects.py
    
Output:
    - zhu_il2_trans_effects.csv
    - zhu_ifng_trans_effects.csv
"""

import h5py
import pandas as pd
import numpy as np
import os

# Configuration
H5AD_PATH = '/home/ec2-user/scitoseq/data/GWCD4i.DE_stats.h5ad'
OUTPUT_DIR = '/home/ec2-user/scitoseq/data'

def extract_gene_trans_effects(filepath, target_gene_name, target_ensembl_id=None):
    """
    Extract trans-effects for a specific target gene from Zhu h5ad file.
    
    Parameters
    ----------
    filepath : str
        Path to the h5ad file
    target_gene_name : str
        Gene symbol (e.g., 'IL2', 'IFNG')
    target_ensembl_id : str, optional
        ENSEMBL ID as fallback
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: perturbation_id, perturbed_gene, condition,
        target_lfc, target_zscore, target_adj_pval
    """
    
    with h5py.File(filepath, 'r') as f:
        
        # Get gene names and IDs from var
        gene_names = [x.decode() if isinstance(x, bytes) else x 
                      for x in f['var']['gene_name'][:]]
        gene_ids = [x.decode() if isinstance(x, bytes) else x 
                    for x in f['var']['_index'][:]]
        
        print(f"Total measured genes: {len(gene_names)}")
        
        # Find target gene index
        target_idx = None
        if target_gene_name in gene_names:
            target_idx = gene_names.index(target_gene_name)
            print(f"{target_gene_name} found at index {target_idx}")
            print(f"{target_gene_name} ENSEMBL ID: {gene_ids[target_idx]}")
        elif target_ensembl_id and target_ensembl_id in gene_ids:
            target_idx = gene_ids.index(target_ensembl_id)
            print(f"{target_gene_name} ({target_ensembl_id}) found at index {target_idx}")
        else:
            print(f"ERROR: {target_gene_name} not found!")
            return None
            
        # Get perturbation info from obs
        obs_index = [x.decode() if isinstance(x, bytes) else x 
                     for x in f['obs']['index'][:]]
        
        # Extract target gene column from layers
        print(f"\nExtracting {target_gene_name} trans-effects...")
        target_lfc = f['layers']['log_fc'][:, target_idx]
        target_zscore = f['layers']['zscore'][:, target_idx]
        target_pval = f['layers']['adj_p_value'][:, target_idx]
        
        # Create dataframe
        df = pd.DataFrame({
            'perturbation_id': obs_index,
            f'{target_gene_name.lower()}_lfc': target_lfc,
            f'{target_gene_name.lower()}_zscore': target_zscore,
            f'{target_gene_name.lower()}_adj_pval': target_pval
        })
        
        # Parse perturbation ID to get gene and condition
        # Format: ENSEMBL_ID_Condition (e.g., ENSG00000123456_Rest)
        df['perturbed_gene'] = df['perturbation_id'].str.replace(
            '_Rest|_Stim8hr|_Stim48hr', '', regex=True
        )
        df['condition'] = df['perturbation_id'].str.extract('_(Rest|Stim8hr|Stim48hr)$')[0]
        
        print(f"\nExtracted {len(df)} trans-effects on {target_gene_name}")
        print(f"Conditions: {df['condition'].value_counts().to_dict()}")
        
        # Summary statistics
        for cond in df['condition'].unique():
            subset = df[df['condition'] == cond]
            col = f'{target_gene_name.lower()}_lfc'
            print(f"\n{cond}: LFC range [{subset[col].min():.3f}, {subset[col].max():.3f}]")
            print(f"  Non-zero effects: {(subset[col].abs() > 0.1).sum()}")
            print(f"  Significant (adj_p < 0.05): {(subset[f'{target_gene_name.lower()}_adj_pval'] < 0.05).sum()}")
        
        return df


def main():
    """Main function to extract IL2 and IFNG trans-effects."""
    
    print("=" * 70)
    print("EXTRACTING TRANS-EFFECTS FROM ZHU ET AL. PERTURB-SEQ DATA")
    print("=" * 70)
    
    # Check if h5ad file exists
    if not os.path.exists(H5AD_PATH):
        print(f"\nERROR: h5ad file not found at {H5AD_PATH}")
        print("Please update H5AD_PATH variable to point to the correct location.")
        return
    
    # First, let's explore the available genes
    print("\n" + "=" * 70)
    print("EXPLORING AVAILABLE GENES")
    print("=" * 70)
    
    with h5py.File(H5AD_PATH, 'r') as f:
        gene_names = [x.decode() if isinstance(x, bytes) else x 
                      for x in f['var']['gene_name'][:]]
        
        # Check for cytokine genes
        cytokines_of_interest = ['IL2', 'IFNG', 'IL2RA', 'FOXP3', 'TNF', 'IL10', 'IL4', 'IL17A']
        print("\nCytokine/marker genes in dataset:")
        for gene in cytokines_of_interest:
            if gene in gene_names:
                print(f"  ✓ {gene}")
            else:
                print(f"  ✗ {gene} (not found)")
    
    # Extract IL2 trans-effects
    print("\n" + "=" * 70)
    print("SECTION 1: EXTRACTING IL2 TRANS-EFFECTS")
    print("=" * 70)
    
    il2_df = extract_gene_trans_effects(
        H5AD_PATH, 
        target_gene_name='IL2',
        target_ensembl_id='ENSG00000109471'  # IL2 ENSEMBL ID
    )
    
    if il2_df is not None:
        output_path = os.path.join(OUTPUT_DIR, 'zhu_il2_trans_effects.csv')
        il2_df.to_csv(output_path, index=False)
        print(f"\nSaved IL2 trans-effects to: {output_path}")
        
        # Preview top regulators
        print("\nTop 10 positive IL2 regulators (Rest condition, by z-score):")
        rest_df = il2_df[il2_df['condition'] == 'Rest'].copy()
        print(rest_df.nsmallest(10, 'il2_zscore')[
            ['perturbed_gene', 'il2_lfc', 'il2_zscore', 'il2_adj_pval']
        ].to_string(index=False))
    
    # Extract IFNG trans-effects
    print("\n" + "=" * 70)
    print("SECTION 2: EXTRACTING IFNG TRANS-EFFECTS")
    print("=" * 70)
    
    ifng_df = extract_gene_trans_effects(
        H5AD_PATH, 
        target_gene_name='IFNG',
        target_ensembl_id='ENSG00000111537'  # IFNG ENSEMBL ID
    )
    
    if ifng_df is not None:
        output_path = os.path.join(OUTPUT_DIR, 'zhu_ifng_trans_effects.csv')
        ifng_df.to_csv(output_path, index=False)
        print(f"\nSaved IFNG trans-effects to: {output_path}")
        
        # Preview top regulators
        print("\nTop 10 positive IFNG regulators (Stim8hr condition, by z-score):")
        stim_df = ifng_df[ifng_df['condition'] == 'Stim8hr'].copy()
        print(stim_df.nsmallest(10, 'ifng_zscore')[
            ['perturbed_gene', 'ifng_lfc', 'ifng_zscore', 'ifng_adj_pval']
        ].to_string(index=False))
    
    # Also extract expression-matched random genes for control comparisons
    print("\n" + "=" * 70)
    print("SECTION 3: EXTRACTING CONTROL GENES FOR RANDOM COMPARISONS")
    print("=" * 70)
    
    # These should be genes with similar expression levels but different biology
    # For a proper control, you'd need to match expression levels to IL2/IFNG
    # For now, let's extract a few housekeeping/unrelated genes
    control_genes = [
        ('ACTB', 'ENSG00000075624'),   # Housekeeping
        ('GAPDH', 'ENSG00000111640'),  # Housekeeping  
        ('CD4', 'ENSG00000010610'),    # Surface marker
    ]
    
    for gene_name, ensembl_id in control_genes:
        print(f"\nExtracting {gene_name}...")
        ctrl_df = extract_gene_trans_effects(
            H5AD_PATH,
            target_gene_name=gene_name,
            target_ensembl_id=ensembl_id
        )
        if ctrl_df is not None:
            output_path = os.path.join(OUTPUT_DIR, f'zhu_{gene_name.lower()}_trans_effects.csv')
            ctrl_df.to_csv(output_path, index=False)
            print(f"Saved to: {output_path}")
    
    print("\n" + "=" * 70)
    print("EXTRACTION COMPLETE")
    print("=" * 70)
    print(f"\nOutput files saved to: {OUTPUT_DIR}")
    print("\nNext step: Run the R script to compare with Schmidt et al. FACS screen")


if __name__ == "__main__":
    main()


EXTRACTING TRANS-EFFECTS FROM ZHU ET AL. PERTURB-SEQ DATA

EXPLORING AVAILABLE GENES

Cytokine/marker genes in dataset:
  ✓ IL2
  ✓ IFNG
  ✓ IL2RA
  ✓ FOXP3
  ✓ TNF
  ✓ IL10
  ✓ IL4
  ✗ IL17A (not found)

SECTION 1: EXTRACTING IL2 TRANS-EFFECTS
Total measured genes: 10282
IL2 found at index 2377
IL2 ENSEMBL ID: ENSG00000109471

Extracting IL2 trans-effects...

Extracted 33983 trans-effects on IL2
Conditions: {'Stim8hr': 11415, 'Rest': 11287, 'Stim48hr': 11281}

Rest: LFC range [-4.071, 5.804]
  Non-zero effects: 10350
  Significant (adj_p < 0.05): 133

Stim48hr: LFC range [-16.121, 6.531]
  Non-zero effects: 10756
  Significant (adj_p < 0.05): 131

Stim8hr: LFC range [-4.621, 4.799]
  Non-zero effects: 10345
  Significant (adj_p < 0.05): 169

Saved IL2 trans-effects to: /home/ec2-user/scitoseq/data/zhu_il2_trans_effects.csv

Top 10 positive IL2 regulators (Rest condition, by z-score):
 perturbed_gene   il2_lfc  il2_zscore  il2_adj_pval
ENSG00000010438 -2.259980   -2.737666      0.99981

In [1]:
#!/usr/bin/env python3
"""
Extract trans-effects from Zhu et al. DE stats for all genes 
that correspond to measured protein markers (self-targeting pairs).

Input:
  - guide_to_marker_mapping.csv: Contains gene_rep column with 196 gene-marker pairs
  - GWCD4i.DE_stats.h5ad: Zhu et al. differential expression results

Output:
  - Individual CSVs per gene: zhu_{gene}_trans_effects.csv
  - Combined summary: zhu_all_marker_genes_trans_effects.csv
"""

import h5py
import pandas as pd
import numpy as np
import os

# === Config ===
H5AD_PATH = '/home/ec2-user/scitoseq/data/GWCD4i.DE_stats.h5ad'
MAPPING_PATH = '/home/ec2-user/scitoseq/data/guide_to_marker_mapping.csv'
OUTPUT_DIR = '/home/ec2-user/scitoseq/data/zhu_trans_effects'

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Load guide-to-marker mapping ===
print("Loading guide-to-marker mapping...")
mapping_df = pd.read_csv(MAPPING_PATH)
print(f"  Total rows: {len(mapping_df)}")
print(f"  Columns: {list(mapping_df.columns)}")

# Get unique genes to extract
genes_to_extract = mapping_df['gene_rep'].unique()
print(f"  Unique genes to extract: {len(genes_to_extract)}")

# === Open h5ad and build gene index ===
print("\nOpening Zhu et al. h5ad file...")
with h5py.File(H5AD_PATH, 'r') as f:
    
    # Get gene names and IDs from var
    gene_names = [x.decode() if isinstance(x, bytes) else x for x in f['var']['gene_name'][:]]
    gene_ids = [x.decode() if isinstance(x, bytes) else x for x in f['var']['_index'][:]]
    
    print(f"  Total genes in Zhu data: {len(gene_names)}")
    
    # Build lookup dictionaries
    name_to_idx = {name: idx for idx, name in enumerate(gene_names)}
    id_to_idx = {gid: idx for idx, gid in enumerate(gene_ids)}
    
    # Get perturbation info from obs
    obs_index = [x.decode() if isinstance(x, bytes) else x for x in f['obs']['index'][:]]
    n_perturbations = len(obs_index)
    print(f"  Total perturbations: {n_perturbations}")
    
    # === Extract trans-effects for each gene ===
    results_list = []
    found_genes = []
    missing_genes = []
    
    for gene in genes_to_extract:
        # Try to find gene by name first, then by ENSEMBL ID
        if gene in name_to_idx:
            gene_idx = name_to_idx[gene]
            found_as = 'gene_name'
        elif gene in id_to_idx:
            gene_idx = id_to_idx[gene]
            found_as = 'ensembl_id'
        else:
            # Gene not found in Zhu data
            missing_genes.append(gene)
            continue
        
        found_genes.append(gene)
        
        # Extract data for this gene
        lfc = f['layers']['log_fc'][:, gene_idx]
        zscore = f['layers']['zscore'][:, gene_idx]
        adj_pval = f['layers']['adj_p_value'][:, gene_idx]
        
        # Create dataframe for this gene
        gene_df = pd.DataFrame({
            'perturbation_id': obs_index,
            'target_gene': gene,
            'target_gene_idx': gene_idx,
            'lfc': lfc,
            'zscore': zscore,
            'adj_pval': adj_pval
        })
        
        # Parse perturbation ID to get perturbed gene and condition
        gene_df['perturbed_gene'] = gene_df['perturbation_id'].str.replace(
            '_Rest|_Stim8hr|_Stim48hr', '', regex=True
        )
        gene_df['condition'] = gene_df['perturbation_id'].str.extract(
            '_(Rest|Stim8hr|Stim48hr)$'
        )[0]
        
        # Add to combined results
        results_list.append(gene_df)
        
        # Save individual file
        individual_path = os.path.join(OUTPUT_DIR, f'zhu_{gene}_trans_effects.csv')
        gene_df.to_csv(individual_path, index=False)
    
    print(f"\n  Found {len(found_genes)} genes in Zhu data")
    print(f"  Missing {len(missing_genes)} genes")

# === Report missing genes ===
if missing_genes:
    print(f"\nMissing genes ({len(missing_genes)}):")
    for g in sorted(missing_genes):
        print(f"  - {g}")
    
    # Save missing genes list
    missing_df = pd.DataFrame({'gene': missing_genes})
    missing_df.to_csv(os.path.join(OUTPUT_DIR, 'missing_genes_in_zhu.csv'), index=False)

# === Combine all results ===
print("\nCombining all results...")
if results_list:
    combined_df = pd.concat(results_list, ignore_index=True)
    combined_path = os.path.join(OUTPUT_DIR, 'zhu_all_marker_genes_trans_effects.csv')
    combined_df.to_csv(combined_path, index=False)
    print(f"  Combined shape: {combined_df.shape}")
    print(f"  Saved to: {combined_path}")
    
    # Summary statistics
    print("\n=== Summary ===")
    print(f"Total genes extracted: {len(found_genes)}")
    print(f"Total rows in combined file: {len(combined_df)}")
    print(f"Conditions: {combined_df['condition'].value_counts().to_dict()}")
    
    # Preview: genes with strongest self-effects in Rest condition
    print("\nSample: Top 10 self-targeting effects (Rest, by |LFC|):")
    
    # Create mapping from gene to ENSEMBL ID for self-effect lookup
    gene_to_ensembl = {}
    with h5py.File(H5AD_PATH, 'r') as f:
        for idx, (gname, gid) in enumerate(zip(gene_names, gene_ids)):
            gene_to_ensembl[gname] = gid
    
    # Find self-targeting effects
    self_effects = []
    for gene in found_genes:
        if gene in gene_to_ensembl:
            ensembl_id = gene_to_ensembl[gene]
            # Look for rows where perturbed_gene matches the ENSEMBL ID
            gene_data = combined_df[
                (combined_df['target_gene'] == gene) & 
                (combined_df['perturbed_gene'] == ensembl_id) &
                (combined_df['condition'] == 'Rest')
            ]
            if len(gene_data) > 0:
                row = gene_data.iloc[0]
                self_effects.append({
                    'gene': gene,
                    'lfc': row['lfc'],
                    'zscore': row['zscore'],
                    'adj_pval': row['adj_pval']
                })
    
    if self_effects:
        self_df = pd.DataFrame(self_effects)
        self_df['abs_lfc'] = self_df['lfc'].abs()
        self_df = self_df.sort_values('abs_lfc', ascending=False)
        print(self_df.head(10)[['gene', 'lfc', 'zscore', 'adj_pval']].to_string(index=False))
        
        # Save self-effects summary
        self_df.to_csv(os.path.join(OUTPUT_DIR, 'zhu_self_targeting_effects_rest.csv'), index=False)

print("\nDone!")

Loading guide-to-marker mapping...
  Total rows: 196
  Columns: ['guide', 'gene_rep', 'marker_rep', 'gene_list_str', 'marker_list_str']
  Unique genes to extract: 196

Opening Zhu et al. h5ad file...
  Total genes in Zhu data: 10282
  Total perturbations: 33983

  Found 136 genes in Zhu data
  Missing 60 genes

Missing genes (60):
  - CCR9
  - CD1D
  - CD24
  - CD244
  - CD34
  - CD36
  - CD82
  - CD8B
  - CDH1
  - CDH5
  - CLEC1B
  - CR2
  - CSF1R
  - EGFR
  - FCER2
  - FCGR1B
  - FCGR2A
  - FCGR2B
  - FCGR2C
  - FCGR3A
  - FCGR3B
  - FCRL3
  - FLT3
  - FOLR2
  - GGT1
  - GP1BA
  - HLA-A
  - HLA-B
  - HLA-C
  - HLA-DPA1
  - HLA-DPB1
  - HLA-DQA1
  - HLA-DQB1
  - HLA-DRA
  - HLA-DRB3
  - HLA-DRB5
  - HLA-E
  - ITGAE
  - ITGB4
  - KIR2DL3
  - KIR2DL4
  - KIR2DS4
  - KIR3DL1
  - KIT
  - KLRF1
  - LILRB1
  - MME
  - MR1
  - NCAM1
  - NCR1
  - NCR2
  - NOTCH3
  - SDC1
  - TFRC
  - THY1
  - TNFRSF13B
  - TNFRSF17
  - VCAM1
  - VTCN1
  - XCR1

Combining all results...
  Combined shape: (4621